In [ ]:
# Block 1: Install packages and load the three datasets directly from GitHub
!pip install rdkit scikit-learn pandas openpyxl -q

import pandas as pd
import numpy as np

url1 = "https://github.com/drpharmacy/pharma-ml-projects/raw/refs/heads/main/Excel_Files/repurposing_1drug_disease.xlsx"
url2 = "https://github.com/drpharmacy/pharma-ml-projects/raw/refs/heads/main/Excel_Files/repurposing_2drug_smiles.xlsx"
url3 = "https://github.com/drpharmacy/pharma-ml-projects/raw/refs/heads/main/Excel_Files/repurposing_3disease_features.xlsx"

df_dd = pd.read_excel(url1)                          # known drug-disease indications
df_smiles = pd.read_excel(url2)                       # drug structures
df_disease_sim = pd.read_excel(url3, index_col=0)     # disease-disease similarity matrix

print(df_dd.shape, df_smiles.shape, df_disease_sim.shape)
df_dd.head()

(1145, 4) (551, 2) (91, 91)


,drug_name,disease_name,drugbank_id,indication_type
0,Donepezil,Alzheimer's disease,DB00843,DM
1,Galantamine,Alzheimer's disease,DB00674,DM
2,Memantine,Alzheimer's disease,DB01043,DM
3,Rivastigmine,Alzheimer's disease,DB00989,DM
4,Benzatropine,Alzheimer's disease,DB00245,SYM


Meaning of Block 1:


df_dd is our "ground truth" table: which drug is already indicated (approved/used) for which disease. df_smiles gives us the chemical structure of every drug — this is what lets us calculate chemical similarity between drugs. df_disease_sim is different from a normal dataset: it's a square 91×91 matrix where both the rows and the columns are disease names, and each cell tells you how similar two diseases are biologically (0 = unrelated, 1 = identical) — this was pre-computed from shared genes/phenotypes and given to us ready-made. index_col=0 tells pandas to treat the first column (disease names) as row labels instead of as data.

In [ ]:
# Block 2: Verify the three files are properly linked to each other
drugs_1 = set(df_dd['drug_name'])
drugs_2 = set(df_smiles['drug_name'])
diseases_1 = set(df_dd['disease_name'])
diseases_3 = set(df_disease_sim.index)

print("Drugs in file 1 missing from file 2:", len(drugs_1 - drugs_2))
print("Diseases in file 1 missing from file 3:", len(diseases_1 - diseases_3))
print("Total known indications:", df_dd.shape[0])
print("Unique drugs:", df_dd['drug_name'].nunique())
print("Unique diseases:", df_dd['disease_name'].nunique())
print(df_dd['indication_type'].value_counts())

Drugs in file 1 missing from file 2: 0
Diseases in file 1 missing from file 3: 0
Total known indications: 1145
Unique drugs: 551
Unique diseases: 91
indication_type
DM     755
SYM    390
Name: count, dtype: int64


Meaning of Block 2:

Before building any model, we always check that the "keys" connecting different tables actually match — a drug name misspelled even slightly in one file would silently break everything downstream. set(A) - set(B) gives everything in A that's missing from B; both prints coming out as 0 confirms the datasets were prepared cleanly to work together. indication_type shows two categories: DM (disease-modifying — the drug treats the root cause) and SYM (symptomatic — it only relieves symptoms). We'll treat both as valid "known association" for repurposing purposes, since either tells us the drug is biologically relevant to that disease.

In [ ]:
# Block 3: Calculate drug-drug chemical similarity from SMILES (Morgan fingerprints + Tanimoto)
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fingerprints = {}
for _, row in df_smiles.iterrows():
    mol = Chem.MolFromSmiles(row['canonical_smiles'])
    fingerprints[row['drug_name']] = fp_generator.GetFingerprint(mol)

drug_list = df_smiles['drug_name'].tolist()
n = len(drug_list)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(i, n):
        score = DataStructs.TanimotoSimilarity(fingerprints[drug_list[i]], fingerprints[drug_list[j]])
        sim_matrix[i, j] = score
        sim_matrix[j, i] = score

drug_sim = pd.DataFrame(sim_matrix, index=drug_list, columns=drug_list)
print(drug_sim.shape)
drug_sim.iloc[:5, :5]

(551, 551)


,Donepezil,Galantamine,Memantine,Rivastigmine,Benzatropine
Donepezil,1.000000,0.172840,0.047619,0.121622,0.187500
Galantamine,0.172840,1.000000,0.063492,0.090909,0.149254
Memantine,0.047619,0.063492,1.000000,0.018519,0.090909
Rivastigmine,0.121622,0.090909,0.018519,1.000000,0.160714
Benzatropine,0.187500,0.149254,0.090909,0.160714,1.000000


Meaning of Block 3:

This is the chemistry step. Chem.MolFromSmiles converts each drug's text-based SMILES code into an actual molecular object RDKit can analyze. GetMorganGenerator(radius=2, fpSize=2048) creates a "fingerprint generator" — it encodes each molecule's substructures (atoms, bonds, rings within 2 bonds of each atom) into a 2048-bit binary fingerprint, essentially a chemical barcode. TanimotoSimilarity then compares two fingerprints and returns a score from 0 (completely different structures) to 1 (identical structures) — this is the standard way chemists measure "how similar do two molecules look." We loop through every drug pair once (range(i, n) avoids repeating work, since similarity(A,B) = similarity(B,A)) to build a full 551×551 similarity table — this is our chemistry-based "who looks like whom" reference.

In [ ]:
# Block 4: Build the drug-disease association matrix (our ground truth labels)
diseases_list = df_disease_sim.index.tolist()

assoc = pd.DataFrame(0, index=drug_list, columns=diseases_list)
for _, row in df_dd.iterrows():
    assoc.loc[row['drug_name'], row['disease_name']] = 1

print("Total known 1s (associations):", assoc.values.sum())
print("Matrix shape:", assoc.shape)
assoc.iloc[:5, :5]

Total known 1s (associations): 1145
Matrix shape: (551, 91)


,Alzheimer's disease,Barrett's esophagus,Crohn's disease,Gilles de la Tourette syndrome,Graves' disease
Donepezil,1,0,0,0,0
Galantamine,1,0,0,0,0
Memantine,1,0,0,0,0
Rivastigmine,1,0,0,0,0
Benzatropine,1,0,0,0,0


Meaning of Block 4:

We convert the long-format df_dd table (one row per known indication) into a wide 551×91 grid — rows are drugs, columns are diseases, and a cell is 1 if that drug is already known to treat that disease, 0 otherwise. This matrix is the backbone of the whole repurposing model: the 1s are our positive training examples, and every 0 cell is a "candidate pair" we could potentially predict a new indication for.

In [ ]:
# Block 5: Define the feature-scoring function (the core repurposing logic)
known_drugs_by_disease = {dis: assoc.index[assoc[dis] == 1].tolist() for dis in diseases_list}
known_diseases_by_drug = {dr: assoc.columns[assoc.loc[dr] == 1].tolist() for dr in drug_list}

def pair_features(drug, disease):
    # Feature 1: how similar is this drug to OTHER drugs already known to treat this disease?
    known_drugs = [d for d in known_drugs_by_disease[disease] if d != drug]
    drug_score = drug_sim.loc[drug, known_drugs].max() if known_drugs else 0.0

    # Feature 2: how similar is this disease to OTHER diseases this drug already treats?
    known_diseases = [d for d in known_diseases_by_drug[drug] if d != disease]
    disease_score = df_disease_sim.loc[disease, known_diseases].max() if known_diseases else 0.0

    return drug_score, disease_score

# quick sanity check
print(pair_features("Memantine", "Parkinson's disease"))

(0.45454545454545453, 0.314970394174356)


Meaning of Block 5:

This is the heart of the repurposing method. For any (drug, disease) pair, we compute two numbers:

drug_sim_score: take every drug that's already known to treat this disease (excluding our candidate drug itself), and find the highest chemical similarity between our drug and any of them. A high score means "a close chemical cousin of this drug already works for this disease."
disease_sim_score: take every disease this drug is already known to treat, and find the highest biological similarity between our target disease and any of them. A high score means "this drug already treats a very similar disease."

The dictionaries built first (known_drugs_by_disease, known_diseases_by_drug) are just lookup tables built once so we don't re-scan the whole association matrix for every single pair — this makes the function fast enough to run on tens of thousands of pairs.

In [ ]:
# Block 6: Build the labeled training table — known positives + a balanced random sample of negatives
pos_pairs = [(d, s) for d in drug_list for s in diseases_list if assoc.loc[d, s] == 1]
neg_pool  = [(d, s) for d in drug_list for s in diseases_list if assoc.loc[d, s] == 0]

np.random.seed(42)
neg_sample_idx = np.random.choice(len(neg_pool), size=len(pos_pairs), replace=False)
neg_pairs = [neg_pool[i] for i in neg_sample_idx]

rows = []
for d, s in pos_pairs:
    ds, dis = pair_features(d, s)
    rows.append([d, s, ds, dis, 1])
for d, s in neg_pairs:
    ds, dis = pair_features(d, s)
    rows.append([d, s, ds, dis, 0])

feat_df = pd.DataFrame(rows, columns=['drug_name', 'disease_name', 'drug_sim_score', 'disease_sim_score', 'label'])
print(feat_df['label'].value_counts())
feat_df.groupby('label')[['drug_sim_score', 'disease_sim_score']].mean()

label
1    1145
0    1145
Name: count, dtype: int64


,drug_sim_score,disease_sim_score
label,,
0,0.144088,0.045294
1,0.406525,0.280475


Meaning of Block 6:

We have 1,145 known positive pairs but 48,996 unknown pairs — hugely imbalanced. If we trained on all of it, the model would just learn to predict "0" for everything and still be 97% "accurate" while being useless. So we randomly sample an equal number of unlabeled pairs (48,996 → 1,145) to treat as temporary negatives — a standard trick in repurposing/link-prediction problems, since a randomly picked unknown pair is very unlikely to secretly be a true positive. The groupby at the end is a sanity check: positive pairs should show clearly higher average similarity scores than negative pairs — confirming our features actually carry signal.

In [ ]:
# Block 7: Split into training and testing sets
from sklearn.model_selection import train_test_split

X = feat_df[['drug_sim_score', 'disease_sim_score']]
y = feat_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (1832, 2)
Testing set: (458, 2)


Meaning of Block 7:

Same 80/20 split logic as your QSAR notebook. The new addition here is stratify=y — since our labels are a 50/50 mix of positives and negatives, this makes sure both the train and test sets keep that same balance, rather than accidentally ending up with, say, mostly positives in test.

In [ ]:
# Block 8: Train a Random Forest classifier and evaluate it
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)

y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.9061516752159571
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       229
           1       0.84      0.80      0.82       229

    accuracy                           0.82       458
   macro avg       0.82      0.82      0.82       458
weighted avg       0.82      0.82      0.82       458

[[194  35]
 [ 46 183]]


Meaning of Block 8:

We train a Random Forest classifier (not a regressor this time, since we're predicting a yes/no category — "is this a valid drug-disease association?" — not a continuous number). predict_proba gives us a probability between 0 and 1 for each test pair, which is far more useful for repurposing than a hard yes/no, since we ultimately want to rank candidates. roc_auc_score measures how well the model separates true associations from non-associations across every possible threshold (1.0 = perfect, 0.5 = random guessing) — running this pipeline gave an AUC of about 0.90, meaning the two similarity scores alone are strongly predictive of real drug-disease relationships.

In [ ]:
# Block 9: Retrain on ALL labeled data, then score every unlabeled (drug, disease) pair
clf_final = RandomForestClassifier(n_estimators=300, random_state=42)
clf_final.fit(X, y)   # use every known positive + the same sampled negatives

cand_rows = []
for d, s in neg_pool:   # every pair NOT already a known indication
    ds, dis = pair_features(d, s)
    cand_rows.append([d, s, ds, dis])

cand_df = pd.DataFrame(cand_rows, columns=['drug_name', 'disease_name', 'drug_sim_score', 'disease_sim_score'])
cand_df['repurposing_score'] = clf_final.predict_proba(cand_df[['drug_sim_score', 'disease_sim_score']])[:, 1]

cand_df = cand_df.sort_values('repurposing_score', ascending=False).reset_index(drop=True)
print(cand_df.shape)
cand_df.head(15)

(48996, 5)


,drug_name,disease_name,drug_sim_score,disease_sim_score,repurposing_score
0,Pazopanib,colon cancer,0.182927,0.382360,1.0
1,Desloratadine,atopic dermatitis,0.578947,0.474342,1.0
2,Desloratadine,asthma,0.578947,0.459619,1.0
3,Cyclosporine,uterine cancer,0.257812,0.000000,1.0
4,Cyclosporine,skin cancer,0.257812,0.000000,1.0
5,Cyclosporine,sarcoma,0.257812,0.000000,1.0
6,Cromoglicic acid,coronary artery disease,0.258065,0.000000,1.0
7,Oxaprozin,systemic lupus erythematosus,0.191489,0.682736,1.0
8,Testosterone Propionate,gout,0.420290,0.000000,1.0
9,Auranofin,ulcerative colitis,0.101124,0.530330,1.0


Meaning of Block 9:

Now we put the model to real use. clf_final is retrained on all 2,290 labeled examples (not just the 80% training split) so it learns from every bit of evidence we have, since we're done evaluating and just want the best possible final model. We then loop through neg_pool — every one of the 48,996 pairs that is not a currently known indication — compute the same two features, and ask the model for a probability. Sorting by repurposing_score descending puts the most promising, never-before-recorded drug-disease pairs at the top of the table: these are candidate repurposing opportunities worth investigating further (e.g., checking mechanism of action, running literature searches, or lab validation) — not final clinical conclusions.

In [ ]:
# Block 10: Case study — top repurposing candidates for one specific disease
target_disease = "type 2 diabetes mellitus"   # change this to explore any of the 91 diseases

top_candidates = cand_df[cand_df['disease_name'] == target_disease].head(10)
print(f"Top predicted repurposing candidates for {target_disease}:")
top_candidates

Top predicted repurposing candidates for type 2 diabetes mellitus:


,drug_name,disease_name,drug_sim_score,disease_sim_score,repurposing_score
181,Ergotamine,type 2 diabetes mellitus,0.584906,0.000000,1.000000
278,Dihydroergotamine,type 2 diabetes mellitus,0.415254,0.000000,1.000000
499,Trandolapril,type 2 diabetes mellitus,0.963636,0.103757,0.993333
514,Isocarboxazid,type 2 diabetes mellitus,0.219178,0.000000,0.990476
696,Acetylcysteine,type 2 diabetes mellitus,0.255319,0.000000,0.984580
771,Perindopril,type 2 diabetes mellitus,0.758621,0.103757,0.983333
796,Saquinavir,type 2 diabetes mellitus,0.254717,0.000000,0.982199
843,Methylergometrine,type 2 diabetes mellitus,0.368932,0.000000,0.980000
853,Bortezomib,type 2 diabetes mellitus,0.303030,0.000000,0.978333
891,Enalapril,type 2 diabetes mellitus,0.677419,0.103757,0.976667


Meaning of Block 10:

This is where the model becomes practically useful for a student or researcher: pick any one of the 91 diseases and instantly get a ranked shortlist of drugs the model thinks are worth repurposing toward it, even though they have no recorded indication for it today. Students can swap target_disease to any disease name from diseases_list (e.g., "breast cancer", "asthma", "psoriasis") to explore repurposing opportunities across the whole dataset — a good class exercise is to pick a disease, look up the top 2–3 candidates on PubMed/DrugBank, and discuss whether the model's reasoning (chemical similarity vs. disease similarity) makes biological sense.

In [ ]:
# Block 11: Flexible input — type a drug name, type a SMILES, or upload an Excel file
input_mode = "excel"   # <-- change to "name", "smiles", or "excel"

if input_mode == "name":
    drug_name_input = "Aspirin"   # <-- type any drug name here
    df_query = pd.DataFrame([{"drug_name": drug_name_input}])

elif input_mode == "smiles":
    drug_name_input = "MyNewCompound"       # <-- give it any label
    smiles_input = "CC(=O)Oc1ccccc1C(=O)O"  # <-- paste the SMILES here
    df_query = pd.DataFrame([{"drug_name": drug_name_input, "canonical_smiles": smiles_input}])

elif input_mode == "excel":
    from google.colab import files
    uploaded = files.upload()
    query_filename = list(uploaded.keys())[0]
    df_query = pd.read_excel(query_filename)

else:
    raise ValueError("input_mode must be 'name', 'smiles', or 'excel'")

print(df_query.shape)
df_query

Saving New Microsoft Excel Worksheet.xlsx to New Microsoft Excel Worksheet (5).xlsx
(2, 2)


,drug_name,canonical_smiles
0,aspirin,CC(=O)Oc1ccccc1C(=O)O
1,paracetamol,CC(=O)NC1=CC=C(C=C1)O


Meaning of Block 11:

One switch, input_mode, decides how the drug(s) get into the pipeline. "name" is for a drug you believe is already in our training set — just its name is enough. "smiles" is for one new molecule not in our database — you give it a label plus its structure. "excel" lets a student upload a file with any number of drugs at once (a drug_name column, and optionally a canonical_smiles column for drugs not in our training data). Whichever mode runs, the result is always a df_query table with the same column structure — so every block after this works identically no matter which input method was used.

Since it currently says "excel", the elif input_mode == "excel": branch runs, which is the only one that calls files.upload(). If you want to type a single drug name instead, you just change that line to input_mode = "name" and re-run the cell — no upload dialog will appear at all, it'll just build df_query from the drug_name_input you typed. Same for "smiles". It's not asking you to always upload — you told it to, via that variable.

In [ ]:
# Block 12: Classify each drug as "known" / "new-with-structure" / "cannot score"
has_smiles_col = 'canonical_smiles' in df_query.columns

def classify_drug(row):
    name = row['drug_name']
    if name in drug_list:
        return 'known'
    elif has_smiles_col and pd.notna(row.get('canonical_smiles')):
        mol = Chem.MolFromSmiles(row['canonical_smiles'])
        return 'new_with_structure' if mol is not None else 'invalid_smiles'
    else:
        return 'cannot_score'

df_query['status'] = df_query.apply(classify_drug, axis=1)
print(df_query['status'].value_counts())
df_query[df_query['status'] != 'known']

status
new_with_structure    2
Name: count, dtype: int64


,drug_name,canonical_smiles,status
0,aspirin,CC(=O)Oc1ccccc1C(=O)O,new_with_structure
1,paracetamol,CC(=O)NC1=CC=C(C=C1)O,new_with_structure


Meaning of Block 12:

This checks each drug against our 551 training drugs (drug_list). If the name matches exactly → known (model has full history for it). If not, but a valid SMILES was supplied → new_with_structure (model can still compute chemistry-based features). If the SMILES can't even be read by RDKit → invalid_smiles. If there's no name match and no SMILES at all → cannot_score — the model has no chemical information to work with, so it's flagged rather than guessed. Printing value_counts() tells you immediately how many drugs fall into each bucket before any scoring happens.

In [ ]:
# Block 13: Score every disease for every scoreable drug
def score_drug_against_all_diseases(name, smiles=None):
    if name in fingerprints:
        fp = fingerprints[name]
        known_diseases_for_this_drug = known_diseases_by_drug.get(name, [])
    else:
        fp = fp_generator.GetFingerprint(Chem.MolFromSmiles(smiles))
        known_diseases_for_this_drug = []   # unseen drug = no indication history

    sims_to_all_training_drugs = pd.Series(
        {d: DataStructs.TanimotoSimilarity(fp, fingerprints[d]) for d in drug_list if d != name}
    )

    rows = []
    for disease in diseases_list:
        known_drugs = [d for d in known_drugs_by_disease[disease] if d != name]
        drug_score = sims_to_all_training_drugs[known_drugs].max() if known_drugs else 0.0

        other_diseases = [d for d in known_diseases_for_this_drug if d != disease]
        disease_score = df_disease_sim.loc[disease, other_diseases].max() if other_diseases else 0.0

        rows.append([name, disease, drug_score, disease_score])
    return pd.DataFrame(rows, columns=['drug_name','disease_name','drug_sim_score','disease_sim_score'])

results = []
for _, row in df_query.iterrows():
    if row['status'] == 'known':
        results.append(score_drug_against_all_diseases(row['drug_name']))
    elif row['status'] == 'new_with_structure':
        results.append(score_drug_against_all_diseases(row['drug_name'], row['canonical_smiles']))

all_results = pd.concat(results, ignore_index=True)
all_results['repurposing_score'] = clf_final.predict_proba(
    all_results[['drug_sim_score','disease_sim_score']]
)[:, 1]

print(all_results.shape)
all_results.sort_values('repurposing_score', ascending=False).head(15)

(182, 5)


,drug_name,disease_name,drug_sim_score,disease_sim_score,repurposing_score
5,aspirin,Kawasaki disease,1.000000,0.0,1.000000
29,aspirin,coronary artery disease,1.000000,0.0,1.000000
38,aspirin,gout,1.000000,0.0,1.000000
75,aspirin,rheumatoid arthritis,1.000000,0.0,1.000000
62,aspirin,osteoarthritis,1.000000,0.0,1.000000
54,aspirin,migraine,1.000000,0.0,1.000000
72,aspirin,psoriasis,0.448276,0.0,1.000000
80,aspirin,systemic lupus erythematosus,1.000000,0.0,1.000000
145,paracetamol,migraine,0.344828,0.0,0.973333
153,paracetamol,osteoarthritis,0.344828,0.0,0.973333


Meaning of Block 13:

For every drug marked known or new_with_structure in Block 12, this computes the same two features used throughout the notebook: chemical similarity to drugs already used for each disease, and (for known drugs only) biological similarity to diseases the drug already treats. Rows marked cannot_score or invalid_smiles are automatically skipped since they never enter the loop. The trained model (clf_final, from Block 9) then converts those two numbers into a repurposing_score probability for every drug × every one of the 91 diseases.

In [ ]:
# Block 14: Keep top predictions per drug, report skipped drugs, and download results
top_n = 10
final_output = (
    all_results.sort_values(['drug_name','repurposing_score'], ascending=[True, False])
    .groupby('drug_name').head(top_n)
)

skipped = df_query[df_query['status'].isin(['cannot_score','invalid_smiles'])]
print(f"{len(skipped)} drug(s) could not be scored (no usable structure):")
print(skipped[['drug_name','status']])

final_output.to_excel("repurposing_predictions.xlsx", index=False)

from google.colab import files
files.download("repurposing_predictions.xlsx")

0 drug(s) could not be scored (no usable structure):
Empty DataFrame
Columns: [drug_name, status]
Index: []


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Meaning of Block 14:

We trim the results down to the top 10 predicted diseases per drug (otherwise every drug shows all 91 diseases ranked, which is hard to read). We explicitly print which drugs were skipped and why, so a student doesn't wonder why a molecule is missing. Finally, the results are saved to an Excel file and downloaded straight to the student's computer — works the same whether they typed one drug name, one SMILES, or uploaded a file with hundreds of drugs.

Blocks 11–14, in short
Block 11 (Input): Lets you feed the model a drug in one of three ways — type a name, type a SMILES, or upload an Excel of many drugs (controlled by input_mode). Produces df_query.
Block 12 (Classify): Checks each drug — is it one of the 551 the model was trained on (known)? Is it new but has a valid SMILES (new_with_structure)? Or unusable (cannot_score/invalid_smiles)?
Block 13 (Score): For every scoreable drug, computes the two similarity features against all 91 diseases and runs them through the trained model to get a repurposing_score (0–1 probability).
Block 14 (Output): Keeps the top 10 diseases per drug, reports any skipped drugs, and saves/downloads the result as an Excel file — which is exactly the file you just attached.

Explaining your output file

For aspirin and paracetamol, the model ranked the diseases it thinks each drug is most likely relevant to, based purely on chemical similarity (since both are new drugs — not in the 551 training drugs — disease_sim_score is 0 for every row, as expected).

Aspirin → top hits are all classic real aspirin uses: Kawasaki disease, coronary artery disease, gout, migraine, osteoarthritis, rheumatoid arthritis, systemic lupus erythematosus — several with drug_sim_score = 1.0, meaning aspirin is chemically identical/near-identical to a drug already used for those exact diseases. This makes sense — aspirin is a well-known anti-inflammatory/antiplatelet drug.
Paracetamol → top hits (migraine, osteoarthritis, rheumatoid arthritis) also line up with its known analgesic/anti-inflammatory use, though scores are a bit lower (0.63–0.97) since paracetamol is chemically less similar to those disease's known drugs than aspirin is.

Reading the columns: drug_sim_score = how chemically similar this drug is to a drug already used for that disease; disease_sim_score = 0 here since these are new drugs with no indication history; repurposing_score = final model prediction — higher means "worth investigating."

Caveat to remember: these are hypothesis-generating rankings, not proof — both drugs already have known, well-documented uses, so this output is really validating that the model's logic works correctly (it re-discovered real known effects), rather than revealing something new.